# Lab 4 — Deep Learning Fundamentals with PyTorch

A complete walkthrough of PyTorch basics: tensors, datasets, MLP definition,
training loop, evaluation, and results visualization.

**Dataset**: FashionMNIST (10-class image classification, 28×28 grayscale)

| Section | Topic |
|---------|-------|
| 1 | Tensors |
| 2 | Dataset, DataLoader & Transforms |
| 3 | MLP Model Definition |
| 4 | Loss Function & Optimizer |
| 5 | Training & Evaluation Loops |
| 6 | Full Training Run (10 epochs) |
| 7 | Results Visualization |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms

import matplotlib
matplotlib.use("Agg")  # headless backend — works without display
import matplotlib.pyplot as plt

import numpy as np

print(f"PyTorch {torch.__version__}")
print(f"Torchvision {torchvision.__version__}")

---
## === 1. TENSORS ===

In [ ]:
# ── 1a. Creating tensors ────────────────────────────────────────────

# From a Python list
t_list = torch.Tensor([[1, 2, 3], [4, 5, 6]])
print("From list  :", t_list, "  shape:", t_list.shape)

# Zeros
t_zeros = torch.zeros(2, 4)
print("Zeros      :", t_zeros, "  shape:", t_zeros.shape)

# Random
t_rand = torch.rand(3, 3)
print("Random     :\n", t_rand, "\n  shape:", t_rand.shape)

In [ ]:
# ── 1b. Device transfer ─────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

t_dev = t_rand.to(device)
print(f"Tensor device after .to(): {t_dev.device}")

In [ ]:
# ── 1c. Indexing & slicing ───────────────────────────────────────────

print("Row 0      :", t_rand[0])
print("Element 1,2:", t_rand[1, 2].item())
print("Col slice  :", t_rand[:, 1])  # all rows, column 1

In [ ]:
# ── 1d. Element-wise & matrix operations ────────────────────────────

a = torch.Tensor([[1, 2], [3, 4]])
b = torch.Tensor([[5, 6], [7, 8]])

print("a + b       :\n", a + b)
print("a * b (elem):\n", a * b)
print("a @ b (matmul):\n", torch.matmul(a, b))

---
## === 2. DATASET & DATALOADER ===

In [ ]:
# ── 2a. Transforms pipeline ─────────────────────────────────────────
# ToTensor:  [0,255] uint8  →  [0.0,1.0] float32,  H×W×C → C×H×W
# Normalize: (pixel − 0.5) / 0.5  →  [-1.0, 1.0]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# ── 2b. Download FashionMNIST ────────────────────────────────────────

DATA_DIR = "./data"

train_dataset = datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transform
)
val_dataset = datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transform
)

print(f"Training samples  : {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Image shape       : {train_dataset[0][0].shape}")
print(f"Classes           : {train_dataset.classes}")

In [ ]:
# ── 2c. DataLoaders ──────────────────────────────────────────────────

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

In [ ]:
# ── 2d. Visualise a sample batch (4×4 grid) ─────────────────────────

CLASS_NAMES = train_dataset.classes

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    # De-normalise: [-1,1] → [0,1]
    img = images[i].squeeze() * 0.5 + 0.5
    ax.imshow(img.numpy(), cmap="gray")
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
    ax.axis("off")
fig.suptitle("FashionMNIST — Sample Batch", fontsize=13)
fig.tight_layout()
fig.savefig("sample_batch.png", dpi=120)
plt.show()
print("Saved → sample_batch.png")

---
## === 3. MODEL ===

In [ ]:
class MLP(nn.Module):
    """Multi-Layer Perceptron for FashionMNIST classification.

    Architecture:
        Flatten(28×28) → Linear(784→256) → ReLU
                       → Linear(256→128) → ReLU
                       → Linear(128→10)  (raw logits)
    """

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                    # (B,1,28,28) → (B,784)
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),              # 10 classes, no activation
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: input tensor of shape (B, 1, 28, 28).

        Returns:
            Logits of shape (B, 10).
        """
        return self.net(x)


model = MLP().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

---
## === 4. LOSS & OPTIMIZER ===

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"Loss     : {criterion}")
print(f"Optimizer: {type(optimizer).__name__}  lr={optimizer.defaults['lr']}")

---
## === 5. TRAINING & EVAL LOOPS ===

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """Train the model for one epoch.

    Args:
        model: the neural network.
        loader: training DataLoader.
        criterion: loss function.
        optimizer: optimizer.
        device: computation device.

    Returns:
        (average_loss, accuracy) for the epoch.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Evaluate the model on a dataset.

    Args:
        model: the neural network.
        loader: validation/test DataLoader.
        criterion: loss function.
        device: computation device.

    Returns:
        (average_loss, accuracy).
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


print("Training and evaluation functions defined.")

---
## === 6. TRAINING RUN ===

In [ ]:
NUM_EPOCHS = 10

# Metrics storage
history: dict[str, list[float]] = {
    "train_loss": [],
    "train_acc":  [],
    "val_loss":   [],
    "val_acc":    [],
}

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9}")
print("-" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.4f}% | {val_loss:>10.4f} | {val_acc:>8.4f}%")

print("\nTraining complete.")

---
## === 7. VISUALIZATION ===

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

# ── Plot 1: Loss ────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(epochs_range, history["train_loss"], "o-", label="Train Loss")
ax1.plot(epochs_range, history["val_loss"],   "s-", label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Train vs Validation Loss")
ax1.legend()
ax1.grid(alpha=0.3)
fig1.tight_layout()
fig1.savefig("loss_curve.png", dpi=150)
plt.show()
print("Saved → loss_curve.png")

# ── Plot 2: Accuracy ────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.plot(epochs_range, history["train_acc"], "o-", label="Train Acc")
ax2.plot(epochs_range, history["val_acc"],   "s-", label="Val Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Train vs Validation Accuracy")
ax2.legend()
ax2.grid(alpha=0.3)
fig2.tight_layout()
fig2.savefig("accuracy_curve.png", dpi=150)
plt.show()
print("Saved → accuracy_curve.png")

---
### Summary

| Step | What we did |
|------|-------------|
| 1 | Created, indexed, and operated on **tensors** |
| 2 | Loaded **FashionMNIST** with `ToTensor` + `Normalize`, built `DataLoader`s |
| 3 | Defined a **3-layer MLP** (784 → 256 → 128 → 10) |
| 4 | Set up **CrossEntropyLoss** + **Adam** (lr=1e-3) |
| 5 | Wrote reusable **train** and **evaluate** functions |
| 6 | Trained for **10 epochs** with per-epoch metrics |
| 7 | Plotted **loss** and **accuracy** curves (saved as PNG) |